# Un siècle dans une courbe · *A century in one curve*

Notebook compagnon du chapitre **9. Comprendre le PIB : la mesure de la richesse produite par un pays** — [lire l'article](https://nmlab.io/ressources/comprendre-le-pib).
Companion notebook to chapter **9. Understanding GDP: The Measure of the Wealth a Country Produces** — [read the article](https://nmlab.io/en/ressources/understanding-gdp).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


from pandas import Series

def load_nominal_gdp() -> Series:
    """PIB nominal annuel des États-Unis (GDPA), en dollars courants, en direct de FRED.
    U.S. annual nominal GDP (GDPA), current dollars, live from FRED."""
    return nm.load_fred("GDPA")

gdp = load_nominal_gdp()


import matplotlib.ticker as mticker
import pandas as pd
import matplotlib.dates as mdates
from matplotlib.figure import Figure

LABELS = {
    "fr": dict(
        title="Un siècle dans une courbe",
        sub="PIB nominal des États-Unis, {a}-{b} — échelle logarithmique",
        ylab="milliards de dollars courants (échelle log)",
        kuznets="1934 — le Sénat reçoit\nle rapport Kuznets",
        note="Source : BEA via FRED, série GDPA (dollars courants) — millésime du 15 juillet 2026.\n"
             "Dollars courants : l'inflation explique une large part de la pente — voir le chapitre 10."),
    "en": dict(
        title="A century in one curve",
        sub="U.S. nominal GDP, {a}-{b} — log scale",
        ylab="billions of current dollars (log scale)",
        kuznets="1934 — the Senate receives\nthe Kuznets report",
        note="Source: BEA via FRED, GDPA series (current dollars) — vintage of July 15, 2026.\n"
             "Current dollars: inflation accounts for much of the slope — see chapter 10."),
}

def build_figure(gdp: Series, lang: str) -> Figure:
    """Un siècle de PIB nominal en échelle log, avec trois repères annotés."""
    text = LABELS[lang]
    first_date, first_val = gdp.index[0], gdp.iloc[0]
    last_date, last_val = gdp.index[-1], gdp.iloc[-1]
    k_date = gdp[gdp.index.year == 1934].index[0]
    k_val = gdp[gdp.index.year == 1934].iloc[0]
    multiple = round(last_val / first_val / 100) * 100
    if lang == "fr":
        first_lbl = f"{first_date.year} : {first_val:,.0f} Mds $".replace(",", " ")
        last_lbl = (f"{last_date.year} : {last_val:,.0f} Mds $".replace(",", " ")
                    + f"\n(≈ {multiple:.0f} fois {first_date.year}, en dollars courants)")
    else:
        first_lbl = f"{first_date.year}: ${first_val:,.0f}bn"
        last_lbl = (f"{last_date.year}: ${last_val:,.0f}bn"
                    + f"\n(≈ {multiple:,.0f} times {first_date.year}, in current dollars)")

    fig = nm.figure(height_px=1064)
    ax = nm.axes(fig, left=0.092)
    ax.plot(gdp.index, gdp, color=nm.COLORS["blue"], linewidth=3.2)
    ax.set_yscale("log")
    ax.set_ylim(50, 45000)
    ax.set_yticks([100, 1000, 10000])
    ax.yaxis.set_major_formatter(nm.thousands(lang))
    ax.yaxis.set_minor_locator(mticker.NullLocator())
    ax.set_ylabel(text["ylab"])
    ax.margins(x=0.01)
    ticks = [pd.Timestamp(f"{y}-01-01") for y in (1930, 1950, 1970, 1990, 2010)]
    ticks.append(last_date)
    ax.set_xticks(ticks)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_title("")
    ax.annotate(text["kuznets"], xy=(k_date, k_val), xytext=(48, 168),
                textcoords="offset points", ha="left", va="center", fontsize=22,
                color=nm.COLORS["text"], linespacing=1.5,
                arrowprops=dict(arrowstyle="-", color=nm.COLORS["muted"], lw=1.3))
    ax.annotate(first_lbl, xy=(first_date, first_val), xytext=(78, -46),
                textcoords="offset points", ha="left", va="center", fontsize=22,
                color=nm.COLORS["text"],
                arrowprops=dict(arrowstyle="-", color=nm.COLORS["muted"], lw=1.3))
    ax.annotate(last_lbl, xy=(last_date, last_val), xytext=(-548, -120),
                textcoords="offset points", ha="left", va="center", fontsize=22,
                color=nm.COLORS["text"], linespacing=1.5,
                arrowprops=dict(arrowstyle="-", color=nm.COLORS["muted"], lw=1.3))
    nm.header(fig, text["title"], text["sub"].format(a=first_date.year, b=last_date.year))
    nm.footer(fig, text["note"])
    return fig

build_figure(gdp, LANG)